# 0. Set up your environment

Run the cells in order with the **Bash** kernel. To select it, click **Select Kernel** at the top right (or the kernel name there, such as `base (Python 3.x)`), then **Select Another Kernel...** > **Jupyter Kernel...** > **Bash**. If a cell fails with `SyntaxError`, it ran in Python: switch to Bash and run it again. If **Bash** isn't listed, run `scripts/install-kernel.sh` in the terminal and reload the window (Command Palette > **Developer: Reload Window**).

The kernel is one shell, so variables and functions carry over from cell to cell. Cells that wait for the cluster print their progress and stop on their own; to stop one early, use the Interrupt button. Where the page offers options, run one of them only, so don't use Run All.

This notebook starts at step 2. You did step 1 in the terminal: you cloned the repository and ran `scripts/install-kernel.sh`.


In [ ]:
export AWS_PAGER=""
cd ~/hyperpod-inference-dive-deep

Run this page once, before the modules. Step 1 runs in the Code Editor terminal. After it, you can run the rest of this page, and every module, either in the terminal or as a notebook.

## Prerequisites

- A HyperPod EKS cluster with a GPU instance group of **2 x `ml.g5.2xlarge`** (1 x NVIDIA A10G 24 GB, 8 vCPU, 32 GiB each) and a CPU instance group (`ml.c5.4xlarge` or larger). Every model in this section fits on one A10G, and each replica takes a whole node. The cluster from the SageMaker HyperPod EKS workshop matches
- The **`amazon-sagemaker-hyperpod-inference`** EKS add-on. `scripts/setup.sh` installs it (see below)
- `kubectl`, `aws` CLI v2, `jq`, and `helm` in your terminal. The workshop's Code Editor has all four
- No Hugging Face account or token: every lab uses ungated models
- Module 6 only: the HyperPod task governance add-on, with the cluster policy and compute allocations from the workshop's task governance section


## 2. Free the GPUs

Free the GPUs that the earlier inference labs and the training job may still hold. Each command is safe to run when the resource is already gone:

In [ ]:
kubectl delete jumpstartmodel --all -n default
kubectl delete inferenceendpointconfig --all -n default
kubectl delete hyperpodpytorchjob --all -n default
kubectl get pods -n default

## 3. Prepare the cluster

When no model or training pods are left, run the setup script. It takes about 12 minutes, prints its progress, and is safe to rerun. `-y` skips its confirmation prompt, which a notebook cell can't answer; this is what it does:

| Step | Why |
|---|---|
| Replaces a Helm-installed inference operator with the `amazon-sagemaker-hyperpod-inference` EKS add-on (operator v3.7) | The workshop cluster runs operator v2.2, which can't load models from Hugging Face and has no request limits or caching. The Helm release owns the operator's CRDs, so the script stops if any inference deployment still exists |
| Creates the add-on's IAM roles and an S3 bucket for TLS certificates | The add-on requires them. The operator only uses buckets whose names start with `hyperpod-tls` |
| Runs the add-on's controllers and `metrics-server` on the CPU instance group | Leaves the GPU nodes' few pod slots for models |
| Creates an S3 bucket for model weights (module 2), an ECR repository, and an AWS CodeBuild project (module 3) | The Code Editor can't run Docker, so module 3 builds its image in CodeBuild |
| Writes `env.sh` | The variables every module uses |
| Installs the Bash kernel for Jupyter (again, if step 1 did) | The notebooks run in it |

In [ ]:
cd ~/hyperpod-inference-dive-deep
scripts/setup.sh -y

Without `-y`, the script lists the account, cluster, and resource names it will use, and asks before it changes anything.

## 4. Load the variables

Load the variables. Run this again in every new terminal (each module's notebook does it in its first cell):

In [ ]:
source ~/hyperpod-inference-dive-deep/env.sh
env | grep -E 'REGION|CLUSTER|ACCOUNT_ID|INSTANCE_TYPE|NAMESPACE|TLS_BUCKET|EXECUTION_ROLE|MODEL_BUCKET|ECR_REPOSITORY|BUILD_PROJECT'

Confirm that the add-on is healthy and that your GPU nodes are visible:

In [ ]:
aws eks describe-addon --cluster-name $EKS_CLUSTER_NAME \
  --addon-name amazon-sagemaker-hyperpod-inference --region $REGION \
  --query "addon.{Version:addonVersion,Status:status,Health:health.issues}" --output table

kubectl get nodes -L node.kubernetes.io/instance-type

## Optional: Prometheus metrics

Module 4's Prometheus trigger and module 5's metrics queries need the HyperPod observability add-on and an Amazon Managed Service for Prometheus workspace. To set them up (about 3 minutes; without `-y` the script asks first):

In [ ]:
~/hyperpod-inference-dive-deep/scripts/observability.sh -y

> **Warning:** The observability add-on runs three DaemonSet pods on every node, at a priority that preempts anything else on a full node. Run `scripts/setup.sh` first, and install observability when no model is running. Amazon Managed Grafana isn't set up: it needs IAM Identity Center for sign-in, which workshop accounts don't have.

> **Note:** `kubectl explain` is the source of truth for every field in this section. If a field here differs from your operator version, run `kubectl explain inferenceendpointconfig.spec --recursive` (or `jumpstartmodel.spec`) and trust what your cluster reports.

## Next

Continue with [1. Inference Operator internals](01-operator-internals.ipynb).